# W10〜W11 連敗原因分析

MomentumStrategy(threshold=3.0, window=20) / BTC/JPY 15min 足

W09: +7.42%, 勝率67%　→　W10: -2.18%, 勝率0%　→　W11: -6.17%, 勝率10%

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

from cryptbot.data.normalizer import normalize
from cryptbot.strategies.regime import RegimeDetector, Regime
from cryptbot.strategies.momentum import MomentumStrategy

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'

DATA_DIR = ROOT / 'data' / 'btc_jpy' / '15min'

WINDOWS = {
    'W09': ('2025-03-01', '2025-07-01'),
    'W10': ('2025-07-01', '2025-11-01'),
    'W11': ('2025-11-01', '2026-03-01'),
    'W12': ('2026-03-01', '2026-04-20'),
}

WINDOW_COLORS = {
    'W09': '#2196F3',
    'W10': '#F44336',
    'W11': '#FF9800',
    'W12': '#4CAF50',
}

THRESHOLD = 3.0
MOMENTUM_WINDOW = 20

print('Setup complete')

In [ ]:
# Section 1: データロード
def load_window_data(year_files: list[str]) -> pd.DataFrame:
    """複数 parquet を結合して normalize() 済み DataFrame を返す。"""
    frames = []
    for f in year_files:
        path = DATA_DIR / f
        if path.exists():
            frames.append(pd.read_parquet(path))
    df = pd.concat(frames).sort_values("timestamp").reset_index(drop=True)
    return normalize(df, momentum_window=MOMENTUM_WINDOW)

# W09〜W12 をカバーする年ファイル
raw_full = load_window_data(["2025.parquet", "2026.parquet"])

# 窓ごとにスライス
def slice_window(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    mask = (df["timestamp"] >= pd.Timestamp(start, tz="Asia/Tokyo")) & (df["timestamp"] < pd.Timestamp(end, tz="Asia/Tokyo"))
    return df[mask].copy()

window_data = {
    name: slice_window(raw_full, s, e)
    for name, (s, e) in WINDOWS.items()
}

print("=== データ確認 ===")
for name, df in window_data.items():
    print(f"{name}: {len(df):>5} bars  "
          f"{df['timestamp'].iloc[0]:%Y-%m-%d} 〜 {df['timestamp'].iloc[-1]:%Y-%m-%d}  "
          f"NaN rows: {df['momentum'].isna().sum()}")